# task 2 : Bert

Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os

 Load Data

In [2]:
df = pd.read_csv("../data/raw/Reviews.csv")

# Select Important Columns ONLY

In [3]:
df = df[['ProductId', 'UserId', 'Score', 'Text']]

df.rename(columns={
    'ProductId': 'productId',
    'UserId': 'userId',
    'Score': 'rating',
    'Text': 'text'
}, inplace=True)

Clean Data

In [4]:
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

# Filter Active Users : for quality

In [5]:
user_counts = df['userId'].value_counts()
active_users = user_counts[user_counts >= 3].index
df = df[df['userId'].isin(active_users)]

# SENTIMENT LABELS

In [6]:
def get_sentiment(score):
    if score >= 4:
        return "positive"
    elif score == 3:
        return "neutral"
    else:
        return "negative"

df['sentiment'] = df['rating'].apply(get_sentiment)

# Clean Reviews Dataset

In [7]:
cleaned_reviews = df[['userId', 'productId', 'text', 'rating', 'sentiment']]

save cleaned reviews

In [8]:
os.makedirs("../data/processed", exist_ok=True)
cleaned_reviews.to_csv("../data/processed/cleaned_reviews.csv", index=False)

 # Product Table : for recommendation

In [9]:
product_table = df.groupby('productId').agg({
    'rating': ['mean', 'count']
}).reset_index()

product_table.columns = ['productId', 'avg_rating', 'rating_count']


Save product table

In [10]:
product_table.to_csv("../data/processed/products.csv", index=False)

 Transaction Dataset : for recommendation 

In [ ]:
transactions = df.groupby('userId')['productId'].apply(list).reset_index()
transactions.columns = ['userId', 'items']

Save transactions

In [12]:
transactions.to_csv("../data/processed/transactions.csv", index=False)

check final dataset

In [13]:
print("Dataset Size:", df.shape)
print("Unique Users:", df['userId'].nunique())
print("Unique Products:", df['productId'].nunique())
print("Missing Values:\n", df.isnull().sum())

Dataset Size: (326447, 5)
Unique Users: 47995
Unique Products: 45911
Missing Values:
 productId    0
userId       0
rating       0
text         0
sentiment    0
dtype: int64
